## 1. 환경 설치

In [ ]:
%%bash
set -e

# 핵심 패키지 (필수)
pip install -q easyocr lmdb editdistance
pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

# deep-text-recognition-benchmark: 이미 있으면 pull, 없으면 clone
if [ -d /content/dtrb/.git ]; then
    git -C /content/dtrb pull -q || true
else
    git clone -q https://github.com/clovaai/deep-text-recognition-benchmark /content/dtrb || true
fi
pip install -q -r /content/dtrb/requirements.txt 2>/dev/null || true

# 한국어 폰트 설치
apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true
fc-cache -fv > /dev/null 2>&1 || true

echo "설치 완료 (PIL 합성 데이터 모드)"

## 2. 라이브러리 임포트 / 디바이스 설정

In [ ]:
import sys, os, re, random, shutil, subprocess, textwrap
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

# sys.path에 DTRB 추가 (Colab)
DTRB = '/content/dtrb'
if DTRB not in sys.path:
    sys.path.insert(0, DTRB)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. 경로 & 하이퍼파라미터 설정

In [ ]:
# ─── 데이터 경로 ───────────────────────────────────────────────
DATA_ROOT  = '/content/ocr_data'
TRAIN_DIR  = f'{DATA_ROOT}/train'
VAL_DIR    = f'{DATA_ROOT}/val'

# ─── LMDB 저장 경로 (/content 권장: Colab에서 /tmp 대비 안정적) ──
# subdir=False로 단일 mdb 파일로 저장
LMDB_TRAIN = '/content/lmdb_train.mdb'
LMDB_VAL   = '/content/lmdb_val.mdb'

# ─── 모델 저장 ─────────────────────────────────────────────────
SAVE_DIR     = '/content/checkpoints'
BEST_MODEL   = f'{SAVE_DIR}/best.pth'
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 하이퍼파라미터 ────────────────────────────────────────────
@dataclass
class Config:
    img_h:        int   = 32          # 입력 이미지 높이
    img_w:        int   = 100         # 입력 이미지 너비
    batch_size:   int   = 64
    num_workers:  int   = 2
    lr:           float = 1e-4
    epochs:       int   = 30
    grad_clip:    float = 5.0
    # EasyOCR 언어 (한국어: 'ko', 영어: 'en')
    lang:         str   = 'ko'
    # 사전 학습 모델 가중치 파일명 (easyocr 캐시에 자동 다운로드)
    pretrained:   str   = 'korean_g2'  # ~/.EasyOCR/model/{pretrained}.pth

CFG = Config()
print(CFG)

## 4. 합성 데이터 생성 (데이터가 없을 때)

> 실제 이미지 데이터가 있으면 이 셀은 건너뛰고 `labels.txt` 포맷만 맞춰 주세요.

기본은 **PIL 합성 모드**로 동작합니다 (의존성 충돌 없음).

### 데이터 수집 방법 (3가지 선택지)

| 방법 | 특징 | 권장 |
|---|---|---|
| **① 합성 생성 (이 셀)** | 코드 한 번 실행으로 수천 장 생성, 무료 | 시작 단계 ✓ |
| **② AIHub 공개 데이터** | 실제 문서/도로 한국어 이미지, 무료 신청 | 품질 ↑ |
| **③ 직접 촬영·크롭** | 특정 도메인(영수증, 간판 등) 특화 | 도메인 특화 |

---
**AIHub 추천 데이터셋**
- [야외 실제 촬영 한글 이미지](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=realm&dataSetSn=105)
- [금융 분야 문서 OCR](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=realm&dataSetSn=71521)
- [문자인식 데이터 (텍스트 크롭)](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=realm&dataSetSn=198)

In [ ]:
import glob
from PIL import Image as PILImage, ImageDraw, ImageFont

# ─── 학습에 사용할 한국어 단어/문장 말뭉치 ─────────────────────
CORPUS_KO = [
    '안녕하세요', '감사합니다', '죄송합니다', '반갑습니다', '어서오세요',
    '주문하기', '결제완료', '배송중', '품절', '할인가',
    '서울특별시', '강남구', '홍대입구', '신촌역', '종로3가',
    '010-1234-5678', '02-999-0000', '2024년 3월', '₩15,000',
    '오후 3시30분', 'A동 301호', '1층 안내데스크',
    'OPEN 10:00~22:00', 'Wi-Fi Password', 'Floor B1', 'EXIT',
    'CCTV 작동중', 'No Smoking', 'PUSH', 'PULL',
]


def generate_synthetic_dataset(out_dir: str, texts: list, count: int, img_h: int = 64) -> None:
    os.makedirs(out_dir, exist_ok=True)
    labels = []
    font_paths = glob.glob('/usr/share/fonts/truetype/nanum/*.ttf')

    for i in range(count):
        text = random.choice(texts)
        width = random.randint(140, 280)
        height = img_h
        image = PILImage.new('RGB', (width, height), color=(255, 255, 255))
        drawer = ImageDraw.Draw(image)

        if font_paths:
            font_path = random.choice(font_paths)
            font_size = random.randint(22, 34)
            try:
                font = ImageFont.truetype(font_path, font_size)
            except Exception:
                font = ImageFont.load_default()
        else:
            font = ImageFont.load_default()

        offset_x = random.randint(3, 12)
        offset_y = random.randint(2, 12)
        drawer.text((offset_x, offset_y), text, fill=(0, 0, 0), font=font)

        fname = f'syn_{i:05d}.jpg'
        image.save(os.path.join(out_dir, fname), quality=95)
        labels.append(f'{fname}\t{text}')

        if (i + 1) % 200 == 0:
            print(f'  {i + 1}/{count} 생성 중...')

    with open(os.path.join(out_dir, 'labels.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(labels))
    print(f'{out_dir}: {len(labels)}장 완료 (PIL 모드)')


# ─── 생성 실행 ────────────────────────────────────────────────
if not os.path.exists(TRAIN_DIR) or not os.listdir(TRAIN_DIR):
    print('합성 데이터 생성 시작...')
    generate_synthetic_dataset(TRAIN_DIR, CORPUS_KO, count=3000)
    generate_synthetic_dataset(VAL_DIR, CORPUS_KO, count=300)
else:
    print('기존 데이터 사용')
    label_file = os.path.join(TRAIN_DIR, 'labels.txt')
    if os.path.exists(label_file):
        with open(label_file, encoding='utf-8') as f:
            lines = f.readlines()
        if lines:
            print(f'  총 {len(lines)}장  샘플: {lines[0].strip()}')

# 샘플 이미지 시각화
import matplotlib.pyplot as plt

sample_labels_path = os.path.join(TRAIN_DIR, 'labels.txt')
with open(sample_labels_path, encoding='utf-8') as f:
    samples = [line.strip().split('\t', 1) for line in f.readlines()[:6] if '\t' in line]

fig, axes = plt.subplots(2, 3, figsize=(12, 4))
for ax, (fname, label) in zip(axes.flat, samples):
    image = PILImage.open(os.path.join(TRAIN_DIR, fname))
    ax.imshow(image)
    ax.set_title(label, fontsize=9)
    ax.axis('off')
plt.suptitle('합성 이미지 샘플 (학습 데이터)')
plt.tight_layout()
plt.show()

## 5. LMDB 변환

이미지 폴더 → LMDB 포맷으로 변환합니다.  
Colab 안정성을 위해 `/content/*.mdb` 단일 파일(`subdir=False`)로 생성합니다.

In [ ]:
_BUILDER_SCRIPT = textwrap.dedent("""
import sys, os, lmdb

src_dir, lmdb_path = sys.argv[1], sys.argv[2]

label_file = os.path.join(src_dir, 'labels.txt')
if not os.path.exists(label_file):
    raise RuntimeError(f'labels.txt 없음: {label_file}')

with open(label_file, encoding='utf-8') as f:
    pairs = [line.strip().split('\\t', 1) for line in f if '\\t' in line]

if len(pairs) == 0:
    raise RuntimeError(f'유효한 라벨이 없습니다: {label_file}')

# 전체 이미지 용량 기반 map_size 계산 (여유 4배)
valid_pairs = []
total_bytes = 0
for fname, label in pairs:
    img_path = os.path.join(src_dir, fname)
    if not os.path.exists(img_path):
        continue
    size = os.path.getsize(img_path)
    total_bytes += size
    valid_pairs.append((fname, label))

if len(valid_pairs) == 0:
    raise RuntimeError(f'이미지 파일을 찾을 수 없습니다: {src_dir}')

meta_overhead = len(valid_pairs) * 4096
map_size = max(1024 * 1024 * 1024, int((total_bytes + meta_overhead) * 4))

# Colab 안정성: subdir=False 단일 파일 + 동기식 flush
env = lmdb.open(
    lmdb_path,
    subdir=False,
    map_size=map_size,
    writemap=False,
    map_async=False,
    meminit=False,
    metasync=True,
    sync=True,
    lock=True,
)

BATCH = 64
saved = 0
for batch_start in range(0, len(valid_pairs), BATCH):
    batch = valid_pairs[batch_start : batch_start + BATCH]
    with env.begin(write=True) as txn:
        for fname, label in batch:
            idx = saved + 1
            img_path = os.path.join(src_dir, fname)
            with open(img_path, 'rb') as img_f:
                img_bytes = img_f.read()
            txn.put(f'image-{idx:09d}'.encode(), img_bytes)
            txn.put(f'label-{idx:09d}'.encode(), label.encode())
            saved += 1
    print(f'  처리 {min(batch_start + BATCH, len(valid_pairs))}/{len(valid_pairs)}  저장 {saved}', flush=True)

with env.begin(write=True) as txn:
    txn.put(b'num-samples', str(saved).encode())

env.sync()
env.close()

if saved == 0:
    raise RuntimeError('LMDB에 저장된 샘플이 없습니다.')

print(f'완료: {saved}개  map_size={map_size // 1024**2}MB', flush=True)
""")

SCRIPT_PATH = '/tmp/_lmdb_builder.py'
with open(SCRIPT_PATH, 'w') as f:
    f.write(_BUILDER_SCRIPT)

def _clear_lmdb_path(lmdb_path: str) -> None:
    if os.path.isdir(lmdb_path):
        shutil.rmtree(lmdb_path)
    elif os.path.isfile(lmdb_path):
        os.remove(lmdb_path)


def build_lmdb(src_dir: str, lmdb_path: str, desc: str) -> None:
    _clear_lmdb_path(lmdb_path)

    print(f'{desc} LMDB 생성 중... ({lmdb_path})')
    r = subprocess.run(
        [sys.executable, SCRIPT_PATH, src_dir, lmdb_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f'{desc} LMDB 빌드 실패 (exit {r.returncode})\n{r.stdout}')
    print(f'{desc}: {lmdb_path} 완료')

build_lmdb(TRAIN_DIR, LMDB_TRAIN, '학습')
build_lmdb(VAL_DIR,   LMDB_VAL,   '검증')
print('\nLMDB 변환 완료')

## 6. Dataset & DataLoader

In [ ]:
import lmdb
from io import BytesIO

# ─── 문자 사전 ────────────────────────────────────────────────
def build_charset(lmdb_path: str) -> str:
    """LMDB 라벨에서 고유 문자 집합 추출"""
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    with env.begin() as txn:
        n = int(txn.get(b'num-samples').decode())
        chars = set()
        for i in range(1, n + 1):
            label = txn.get(f'label-{i:09d}'.encode()).decode()
            chars.update(label)
    env.close()
    return ''.join(sorted(chars))

CHARSET   = build_charset(LMDB_TRAIN)
N_CLASSES = len(CHARSET) + 1   # +1 for CTC blank
print(f'문자 수: {len(CHARSET)}  예시: {CHARSET[:30]}...')


# ─── LMDB Dataset ────────────────────────────────────────────
class LmdbDataset(Dataset):
    def __init__(self, lmdb_path: str, img_h: int, img_w: int) -> None:
        self._env    = None
        self.path    = lmdb_path
        self.img_h   = img_h
        self.img_w   = img_w
        self.transform = transforms.Compose([
            transforms.Grayscale(1),
            transforms.Resize((img_h, img_w)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        # 샘플 수 확인
        env = lmdb.open(lmdb_path, readonly=True, lock=False)
        with env.begin() as txn:
            self.length = int(txn.get(b'num-samples').decode())
        env.close()

    def _get_env(self) -> lmdb.Environment:
        """워커별 지연 열기 (multiprocessing 안전)"""
        if self._env is None:
            self._env = lmdb.open(
                self.path, readonly=True, lock=False,
                readahead=False, meminit=False,
            )
        return self._env

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, str]:
        env = self._get_env()
        with env.begin(buffers=True) as txn:
            key_img   = f'image-{idx + 1:09d}'.encode()
            key_label = f'label-{idx + 1:09d}'.encode()
            img_buf   = bytes(txn.get(key_img))
            label     = txn.get(key_label).decode()
        image = Image.open(BytesIO(img_buf)).convert('RGB')
        return self.transform(image), label


# ─── 가변 길이 레이블을 배치로 묶기 ──────────────────────────
def collate_fn(batch: List[Tuple]) -> Tuple:
    images, labels = zip(*batch)
    return torch.stack(images), list(labels)


train_ds = LmdbDataset(LMDB_TRAIN, CFG.img_h, CFG.img_w)
val_ds   = LmdbDataset(LMDB_VAL,   CFG.img_h, CFG.img_w)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == 'cuda'),
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == 'cuda'),
)

print(f'학습: {len(train_ds)}장 / {len(train_loader)} 배치')
print(f'검증: {len(val_ds)}장 / {len(val_loader)} 배치')

# 샘플 확인
imgs, labels = next(iter(train_loader))
print(f'배치 이미지: {imgs.shape}  라벨 예시: {labels[:3]}')

## 7. 사전학습 EasyOCR 모델 로드

EasyOCR은 내부적으로 **CRNN** (VGG feature extractor + BiLSTM + CTC) 구조를 사용합니다.  
`easyocr.Reader`를 통해 가중치를 다운로드하고, 내부 PyTorch 모델을 추출합니다.

In [ ]:
import easyocr

# EasyOCR Reader 초기화 → 가중치 자동 다운로드
reader = easyocr.Reader([CFG.lang], gpu=(DEVICE == 'cuda'), verbose=False)

# 인식 모델 추출 (버전별 속성명 차이 대응)
if hasattr(reader, 'recognizer'):
    model = reader.recognizer
elif hasattr(reader, 'model_lang'):
    model = reader.model_lang
else:
    raise RuntimeError('EasyOCR 인식 모델 속성을 찾지 못했습니다. easyocr 버전을 확인하세요.')

model = model.to(DEVICE)

# ─── 분류기(헤드) 교체: 커스텀 문자 수에 맞게 ────────────────
def replace_classifier(model: nn.Module, n_classes: int) -> nn.Module:
    """마지막 Linear 레이어를 새 클래스 수로 교체"""
    target = model.module if hasattr(model, 'module') else model

    if not hasattr(target, 'Prediction') or not hasattr(target.Prediction, 'generator'):
        raise RuntimeError('Prediction.generator 레이어를 찾을 수 없습니다. 모델 구조를 확인하세요.')

    old_gen = target.Prediction.generator
    new_gen = nn.Linear(old_gen.in_features, n_classes)
    target.Prediction.generator = new_gen
    return model

model = replace_classifier(model, N_CLASSES)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'총 파라미터:    {total_params:,}')
print(f'학습 파라미터:  {trainable_params:,}')

## 8. 파인튜닝 파라미터 설정

- **Backbone 레이어 동결**: VGG feature extractor는 동결, BiLSTM + Attention은 학습
- **손실 함수**: CTC Loss
- **Optimizer**: AdamW + CosineAnnealingLR

In [ ]:
# ─── 레이어 선택적 동결 ───────────────────────────────────────
def freeze_backbone(model: nn.Module) -> None:
    """FeatureExtraction(VGG)만 동결, 나머지는 학습"""
    target = model.module if hasattr(model, 'module') else model
    frozen, unfrozen = 0, 0
    for name, param in target.named_parameters():
        if name.startswith('FeatureExtraction'):
            param.requires_grad = False
            frozen += param.numel()
        else:
            param.requires_grad = True
            unfrozen += param.numel()
    print(f'동결: {frozen:,}  학습: {unfrozen:,}')

freeze_backbone(model)

# ─── 문자 → 인덱스 변환 유틸 ─────────────────────────────────
char2idx = {ch: i + 1 for i, ch in enumerate(CHARSET)}   # 0 = CTC blank
idx2char = {v: k for k, v in char2idx.items()}

def encode_labels(labels: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
    """라벨 문자열 → CTC 입력 형식"""
    encoded, lengths = [], []
    for label in labels:
        enc = [char2idx.get(ch, 0) for ch in label]
        encoded.extend(enc)
        lengths.append(len(enc))
    return (
        torch.tensor(encoded, dtype=torch.long),
        torch.tensor(lengths, dtype=torch.long),
    )

# ─── 손실 / 옵티마이저 / 스케줄러 ───────────────────────────
ctc_loss  = nn.CTCLoss(blank=0, zero_infinity=True).to(DEVICE)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG.lr,
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.epochs, eta_min=1e-6,
)
print('옵티마이저 준비 완료')

## 9. 학습 루프 정의

In [ ]:
def train_one_epoch(model, loader, optimizer, ctc_loss, device, grad_clip):
    model.train()
    total_loss = 0.0

    for imgs, labels in loader:
        imgs = imgs.to(device)

        target_enc, target_len = encode_labels(labels)
        target_enc = target_enc.to(device)
        target_len = target_len.to(device)

        # 순전파: (T, N, C) 형식 출력
        logits = model(imgs, labels, is_train=True)  # (T, N, n_classes)
        log_probs = F.log_softmax(logits, dim=2)

        T, N, _ = log_probs.shape
        input_len = torch.full((N,), T, dtype=torch.long, device=device)

        loss = ctc_loss(log_probs, target_enc, input_len, target_len)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    """CER (Character Error Rate) 계산"""
    model.eval()
    n_correct, n_total = 0, 0
    total_cer = 0.0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs, labels, is_train=False)   # Greedy decode
        _, preds = logits.max(2)
        preds = preds.transpose(1, 0)                  # (N, T)

        for pred_seq, gt in zip(preds, labels):
            # CTC 디코딩 (반복 제거, blank 제거)
            decoded = []
            prev = 0
            for tok in pred_seq.cpu().tolist():
                if tok != 0 and tok != prev:
                    decoded.append(idx2char.get(tok, ''))
                prev = tok
            pred_str = ''.join(decoded)

            n_correct += pred_str == gt
            n_total   += 1
            # 편집 거리 기반 CER
            import editdistance
            cer = editdistance.eval(pred_str, gt) / max(len(gt), 1)
            total_cer += cer

    acc = n_correct / n_total if n_total else 0.0
    cer = total_cer / n_total if n_total else 1.0
    return acc, cer

print('학습 루프 함수 정의 완료')

## 10. 모델 학습

In [ ]:
import time

best_cer   = float('inf')
history    = []

for epoch in range(1, CFG.epochs + 1):
    t0        = time.time()
    train_loss = train_one_epoch(
        model, train_loader, optimizer, ctc_loss, DEVICE, CFG.grad_clip
    )
    val_acc, val_cer = evaluate(model, val_loader, DEVICE)
    scheduler.step()

    elapsed = time.time() - t0
    lr_now  = scheduler.get_last_lr()[0]
    log = {
        'epoch': epoch,
        'loss': round(train_loss, 4),
        'val_acc': round(val_acc, 4),
        'val_cer': round(val_cer, 4),
        'lr': round(lr_now, 7),
    }
    history.append(log)
    print(
        f'[{epoch:3d}/{CFG.epochs}] '
        f'loss={train_loss:.4f}  acc={val_acc:.3f}  '
        f'CER={val_cer:.3f}  lr={lr_now:.2e}  {elapsed:.1f}s'
    )

    # Best 모델 저장
    if val_cer < best_cer:
        best_cer = val_cer
        torch.save(
            {
                'epoch':       epoch,
                'model_state': model.state_dict(),
                'charset':     CHARSET,
                'val_cer':     val_cer,
            },
            BEST_MODEL,
        )
        print(f'  ✓ 베스트 모델 저장 (CER={best_cer:.4f})')

print(f'\n학습 완료  |  Best CER: {best_cer:.4f}')

## 11. 학습 곡선 시각화 & 최종 평가

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Colab 헤드리스 환경
import matplotlib.pyplot as plt

epochs_list = [h['epoch']   for h in history]
losses      = [h['loss']    for h in history]
cers        = [h['val_cer'] for h in history]
accs        = [h['val_acc'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_list, losses, marker='o', markersize=3)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].grid(True)

axes[1].plot(epochs_list, cers, marker='o', markersize=3, color='tomato')
axes[1].set_title('Validation CER (낮을수록 좋음)')
axes[1].set_xlabel('Epoch')
axes[1].grid(True)

axes[2].plot(epochs_list, accs, marker='o', markersize=3, color='seagreen')
axes[2].set_title('Validation Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].grid(True)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curve.png', dpi=150)
plt.show()
print(f'그래프 저장: {SAVE_DIR}/training_curve.png')

# ─── 최종 평가 (베스트 모델 로드) ────────────────────────────
ckpt = torch.load(BEST_MODEL, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
final_acc, final_cer = evaluate(model, val_loader, DEVICE)
print(f'\n[최종 평가] Accuracy: {final_acc:.4f}  CER: {final_cer:.4f}')

## 12. 파인튜닝 모델로 추론

학습된 가중치를 EasyOCR에 교체 후 실제 이미지로 추론합니다.

In [ ]:
def infer_image(img_path: str, model: nn.Module, device: str) -> str:
    """단일 이미지 추론 (파인튜닝 모델 사용)"""
    transform = transforms.Compose([
        transforms.Grayscale(1),
        transforms.Resize((CFG.img_h, CFG.img_w)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    img    = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)  # (1, C, H, W)

    model.eval()
    with torch.no_grad():
        logits = model(tensor, [''], is_train=False)   # (T, 1, n_classes)
        _, preds = logits.max(2)
        preds = preds.squeeze(1)  # (T,)

    decoded, prev = [], 0
    for tok in preds.cpu().tolist():
        if tok != 0 and tok != prev:
            decoded.append(idx2char.get(tok, ''))
        prev = tok
    return ''.join(decoded)


def infer_and_show(img_path: str) -> None:
    result = infer_image(img_path, model, DEVICE)
    img    = Image.open(img_path).convert('RGB')
    plt.figure(figsize=(4, 2))
    plt.imshow(img, cmap='gray')
    plt.title(f'예측: {result}', fontproperties='Malgun Gothic' if os.name=='nt' else None)
    plt.axis('off')
    plt.show()
    print(f'예측 결과: {result}')


# ─── 검증 셋 샘플 몇 개로 테스트 ─────────────────────────────
print('=== 파인튜닝 모델 추론 테스트 ===')
with open(os.path.join(VAL_DIR, 'labels.txt'), encoding='utf-8') as f:
    val_pairs = [line.strip().split('\t', 1) for line in f if '\t' in line]

for fname, gt in random.sample(val_pairs, min(5, len(val_pairs))):
    img_path = os.path.join(VAL_DIR, fname)
    pred     = infer_image(img_path, model, DEVICE)
    match    = '✓' if pred == gt else '✗'
    print(f'  [{match}] GT: {gt:15s}  PRED: {pred}')

## 13. 모델 내보내기 (EasyOCR 교체용)

학습된 가중치를 `~/.EasyOCR/model/` 에 저장하고,  
`easyocr.Reader`의 `model_storage_directory`로 직접 지정해 사용할 수 있습니다.

In [ ]:
import json

# ─── 체크포인트에서 state_dict만 추출해 EasyOCR 형식으로 저장 ─
ckpt       = torch.load(BEST_MODEL, map_location='cpu')
state_dict = ckpt['model_state']

EXPORT_PATH = f'{SAVE_DIR}/finetuned_recognizer.pth'
torch.save(state_dict, EXPORT_PATH)

# 문자 사전 함께 저장 (EasyOCR 커스텀 모델 설정에 필요)
charset_path = f'{SAVE_DIR}/charset.json'
with open(charset_path, 'w', encoding='utf-8') as f:
    json.dump({'charset': CHARSET, 'n_classes': N_CLASSES}, f, ensure_ascii=False)

print(f'모델 저장: {EXPORT_PATH}')
print(f'문자사전:  {charset_path}')

# ─── EasyOCR에서 파인튜닝 모델 사용 예시 ─────────────────────
print("""
[사용 방법]
import easyocr

reader = easyocr.Reader(
    ['ko'],
    recog_network='custom',
    model_storage_directory='/content/checkpoints',   # finetuned_recognizer.pth 위치
    user_network_directory='/content/checkpoints',   # custom_model.py 위치 (선택)
    gpu=True,
)
result = reader.readtext('image.jpg')
""")

# Google Drive 백업 (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE = '/content/drive/MyDrive/easyocr_finetune'
    os.makedirs(GDRIVE, exist_ok=True)
    shutil.copy(EXPORT_PATH, GDRIVE)
    shutil.copy(charset_path, GDRIVE)
    shutil.copy(f'{SAVE_DIR}/training_curve.png', GDRIVE)
    print(f'Google Drive 백업 완료: {GDRIVE}')
except Exception:
    print('(Google Drive 마운트 생략 — 로컬 환경)')